# 🏥 Project 2: Medical Insurance Cost Predictor
### Dataset: Medical Insurance Dataset (insurance.csv)
---

## 📋 Dataset Overview & Expected Target

| Feature | Description | Type |
|---------|-------------|------|
| `age` | Age of the primary beneficiary | Numerical |
| `sex` | Gender: male / female | Categorical (binary) |
| `bmi` | Body Mass Index | Numerical |
| `children` | Number of dependents covered | Numerical (discrete) |
| `smoker` | Whether the person smokes: yes / no | Categorical (binary) |
| `region` | US region: northeast / southeast / southwest / northwest | Categorical |

> 🎯 **Target Variable → `charges`** (continuous, annual insurance cost in USD)  
> 📌 **Task Type → Regression**

> 💡 **Key Insight to Discover:** The `smoker` column has a massive effect on charges — often the single most important feature. The distribution of `charges` is **bimodal** (two humps): one cluster for non-smokers and one for smokers. Spotting this early shapes every modelling decision downstream.


---
## ⚙️ Phase 1 — Environment & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

# sns.set_style("whitegrid")
# plt.rcParams["figure.figsize"] = (10, 6)

TARGET = "charges"

### 1.1 — Load the Dataset

In [ ]:
FILE_PATH = "insurance.csv"    # ← update me

df = pd.read_csv(FILE_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

### 1.2 — Initial Inspection

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# This dataset is famously clean — expect 0 missing values.
print("Missing values per column:")
print(df.isnull().sum())

print("\nUnique values in categorical columns:")
for col in ["sex", "smoker", "region"]:
    print(f"  {col}: {df[col].unique()}")

---
## 📊 Phase 2 — Exploratory Data Analysis (EDA)

### 2.1 — Target Variable Distribution

In [ ]:
# TODO: Plot raw charges on axes[0] and log1p(charges) on axes[1].
#       The raw plot should show a right-skewed, bimodal distribution.

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# axes[0].hist(df[TARGET], bins=50, color="steelblue", edgecolor="white")
# axes[0].set_title("Charges – Raw Distribution")
# axes[0].set_xlabel("Charges (USD)")

# axes[1].hist(np.log1p(df[TARGET]), bins=50, color="darkorange", edgecolor="white")
# axes[1].set_title("Charges – Log-Transformed Distribution")
# axes[1].set_xlabel("log1p(Charges)")

# plt.tight_layout()
# plt.show()

### 2.2 — The SMOKER Effect (Pivotal Plot)

In [ ]:
# TODO: Plot overlapping KDE curves of 'charges' split by smoker status.
#       This single plot reveals the dominant feature of this dataset.

# plt.figure(figsize=(10, 5))
# sns.kdeplot(data=df, x=TARGET, hue="smoker", fill=True, common_norm=False, alpha=0.5)
# plt.title("Charge Distribution: Smoker vs Non-Smoker")
# plt.xlabel("Charges (USD)")
# plt.tight_layout()
# plt.show()

### 2.3 — Correlation Heatmap

In [ ]:
# TODO: Compute and plot the correlation matrix for numerical columns.
#       Which numerical feature correlates most strongly with charges?

# numerical_df = df.select_dtypes(include=[np.number])
# corr_matrix  = numerical_df.corr()

# plt.figure(figsize=(7, 5))
# sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
# plt.title("Correlation Matrix – Numerical Features")
# plt.tight_layout()
# plt.show()

### 2.4 — Pairplot (coloured by Smoker Status)

In [ ]:
# TODO: Run sns.pairplot() with hue="smoker".
#       This is slow but enormously revealing for a small dataset like this.
#       Look for clusters — especially in bmi vs charges.

# sns.pairplot(df, hue="smoker", plot_kws={"alpha": 0.4, "s": 15})
# plt.suptitle("Pairplot – All Features (coloured by Smoker)", y=1.02)
# plt.show()

### 2.5 — BMI vs Charges (coloured by Smoker)

In [ ]:
# TODO: Scatter plot of bmi vs charges, colour by smoker status.
#       You should see two distinct clusters emerge at different BMI thresholds.

# plt.figure(figsize=(9, 5))
# colors = df["smoker"].map({"yes": "tomato", "no": "steelblue"})
# plt.scatter(df["bmi"], df[TARGET], c=colors, alpha=0.5, s=15)
# plt.xlabel("BMI")
# plt.ylabel("Charges (USD)")
# plt.title("BMI vs Charges (red = smoker, blue = non-smoker)")
# plt.tight_layout()
# plt.show()

### 2.6 — Age vs Charges (coloured by Smoker)

In [ ]:
# TODO: Scatter plot of age vs charges, colour by smoker.
#       Three rough bands should emerge:
#         1. Young non-smokers (low cost)
#         2. Older non-smokers (moderate cost)
#         3. Smokers of all ages (high cost)

# plt.figure(figsize=(9, 5))
# colors = df["smoker"].map({"yes": "tomato", "no": "steelblue"})
# plt.scatter(df["age"], df[TARGET], c=colors, alpha=0.5, s=15)
# plt.xlabel("Age")
# plt.ylabel("Charges (USD)")
# plt.title("Age vs Charges (red = smoker, blue = non-smoker)")
# plt.tight_layout()
# plt.show()

### 2.7 — Categorical Breakdowns (Box Plots)

In [ ]:
# TODO: Box plots of charges grouped by: sex | region | children
# for col in ["sex", "region", "children"]:
#     plt.figure(figsize=(9, 4))
#     sns.boxplot(data=df, x=col, y=TARGET, palette="Set2")
#     plt.title(f"Charges by {col}")
#     plt.xticks(rotation=20)
#     plt.tight_layout()
#     plt.show()

### 2.8 — Regional Average Charges

In [ ]:
# TODO: Bar chart of mean charges per region.
#       Are there meaningful geographic pricing differences?

# region_avg = df.groupby("region")[TARGET].mean().sort_values(ascending=False)
# region_avg.plot(kind="bar", figsize=(8, 4), color="coral", edgecolor="white")
# plt.title("Average Charges by Region")
# plt.ylabel("Mean Charges (USD)")
# plt.xticks(rotation=15)
# plt.tight_layout()
# plt.show()

---
## 🔧 Phase 3 — Data Preprocessing & Feature Engineering

In [ ]:
df_clean = df.copy()

### 3.1 — Handle Missing Values

In [ ]:
# This dataset is typically complete. This block is here as good practice.
print("Null counts:")
print(df_clean.isnull().sum())

# TODO: If any nulls exist, add your imputation strategy here.

### 3.2 — Encode Binary Categorical Variables

In [ ]:
# 'sex' and 'smoker' are binary → map directly to 0 / 1.
# TODO: Choose which value maps to 1 (it's your design decision).

# df_clean["sex"]    = df_clean["sex"].map({"female": 0, "male": 1})
# df_clean["smoker"] = df_clean["smoker"].map({"no": 0, "yes": 1})

# Verify the mapping worked
# df_clean[["sex", "smoker"]].value_counts()

### 3.3 — One-Hot Encode 'region'

In [ ]:
# 'region' has 4 categories → use OHE with drop_first=True to avoid multicollinearity.

# df_clean = pd.get_dummies(df_clean, columns=["region"], drop_first=True)
# print("Columns after OHE:", df_clean.columns.tolist())

### 3.4 — Feature Engineering: Interaction Terms

In [ ]:
# The linear model struggles because the smoker/age/bmi interactions are non-linear.
# Creating explicit interaction features helps the linear model capture these effects.

# TODO: Create at minimum these two interaction features:
#   bmi_smoker = bmi × smoker   (captures the dangerous high-BMI + smoking combo)
#   age_smoker = age × smoker

# df_clean["bmi_smoker"] = df_clean["bmi"] * df_clean["smoker"]
# df_clean["age_smoker"] = df_clean["age"] * df_clean["smoker"]

# TODO (optional): Create age² to capture the non-linear age-cost relationship.
# df_clean["age_squared"] = df_clean["age"] ** 2

print("Dataset shape after feature engineering:", df_clean.shape)
df_clean.head()

### 3.5 — (Optional) Target Transformation

In [ ]:
# Apply log1p to charges if the raw distribution is very right-skewed.
# Remember: use np.expm1() on predictions when computing final metrics.

# df_clean[TARGET] = np.log1p(df_clean[TARGET])

print("Final shape:", df_clean.shape)

---
## 🤖 Phase 4 — Model Training & Evaluation

### 4.1 — X / y Split

In [ ]:
# X = df_clean.drop(columns=[TARGET])
# y = df_clean[TARGET]

# print("X shape:", X.shape)
# print("y shape:", y.shape)

### 4.2 — Train / Test Split

In [ ]:
# An 80/20 split works well for this small (~1 338 row) dataset.

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=   ...,
#     random_state= ...,
# )
# print(f"Train: {X_train.shape} | Test: {X_test.shape}")

### 4.3 — (Optional) Feature Scaling

In [ ]:
# Linear / Ridge / Lasso benefit from scaled features.
# Tree models (RF, GB) do NOT need scaling — skip for those.

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled  = scaler.transform(X_test)
# ⚠️  CRITICAL: fit the scaler on X_train ONLY — never on X_test.

### 4.4 — Baseline: Linear Regression

In [ ]:
# lr_model = LinearRegression()
# lr_model.fit(X_train_scaled, y_train)
# lr_preds = lr_model.predict(X_test_scaled)

### 4.5 — Regularised Linear: Ridge Regression

In [ ]:
# Ridge adds an L2 penalty → shrinks coefficients but keeps all features.
# TODO: Try alpha values: 0.1, 1.0, 10, 100.

# ridge_model = Ridge(alpha= ...)
# ridge_model.fit(X_train_scaled, y_train)
# ridge_preds = ridge_model.predict(X_test_scaled)

### 4.6 — Regularised Linear: Lasso Regression

In [ ]:
# Lasso adds an L1 penalty → can zero out feature coefficients entirely (built-in selection).
# TODO: Try alpha values: 0.1, 1.0, 10, 100.

# lasso_model = Lasso(alpha= ...)
# lasso_model.fit(X_train_scaled, y_train)
# lasso_preds = lasso_model.predict(X_test_scaled)

### 4.7 — Ensemble: Random Forest Regressor

In [ ]:
# rf_model = RandomForestRegressor(
#     n_estimators= ...,
#     max_depth=    ...,
#     random_state= ...,
# )
# rf_model.fit(X_train, y_train)    # ← raw (unscaled) X for tree models
# rf_preds = rf_model.predict(X_test)

### 4.8 — Ensemble: Gradient Boosting Regressor

In [ ]:
# gb_model = GradientBoostingRegressor(
#     n_estimators=  ...,
#     learning_rate= ...,
#     max_depth=     ...,
#     random_state=  ...,
# )
# gb_model.fit(X_train, y_train)
# gb_preds = gb_model.predict(X_test)

### 4.9 — Evaluation Helper Function

In [ ]:
def evaluate_model(name: str, y_true, y_pred) -> dict:
    """Compute and display MSE, RMSE, MAE, and R² for a regression model."""
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)

    print(f"\n{'─'*42}")
    print(f"  Model : {name}")
    print(f"{'─'*42}")
    print(f"  MSE   : ${mse:>13,.2f}")
    print(f"  RMSE  : ${rmse:>13,.2f}   ← same unit as target (USD)")
    print(f"  MAE   : ${mae:>13,.2f}")
    print(f"  R²    : {r2:>14.4f}   ← 1.0 = perfect")
    return {"Model": name, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

# TODO: Call evaluate_model() for every fitted model.
# results = []
# results.append(evaluate_model("Linear Regression", y_test, lr_preds))
# results.append(evaluate_model("Ridge",             y_test, ridge_preds))
# results.append(evaluate_model("Lasso",             y_test, lasso_preds))
# results.append(evaluate_model("Random Forest",     y_test, rf_preds))
# results.append(evaluate_model("Gradient Boosting", y_test, gb_preds))

### 4.10 — Model Comparison Table

In [ ]:
# results_df = pd.DataFrame(results).set_index("Model")
# results_df.sort_values("R2", ascending=False)

### 4.11 — Feature Importance (Random Forest)

In [ ]:
# TODO: Is 'smoker' at the top? It almost certainly should be.

# importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
# importances.sort_values().tail(10).plot(
#     kind="barh", figsize=(9, 5), color="steelblue"
# )
# plt.title("Random Forest – Feature Importances")
# plt.xlabel("Importance Score")
# plt.tight_layout()
# plt.show()

### 4.12 — Coefficient Inspection (Linear Model)

In [ ]:
# TODO: Which coefficient is largest? Does it match your EDA findings?

# coef_df = pd.Series(lr_model.coef_, index=X_train.columns).sort_values()
# coef_df.plot(kind="barh", figsize=(9, 5), color="coral")
# plt.title("Linear Regression – Coefficients")
# plt.axvline(0, color="black", linewidth=0.8)
# plt.tight_layout()
# plt.show()

### 4.13 — Actual vs Predicted Plot

In [ ]:
# TODO: Scatter y_test vs best_model_preds.
#       Add a diagonal (y = x) representing perfect predictions.

# plt.figure(figsize=(8, 6))
# plt.scatter(y_test, rf_preds, alpha=0.5, s=15, color="steelblue")
# lims = [min(y_test.min(), rf_preds.min()), max(y_test.max(), rf_preds.max())]
# plt.plot(lims, lims, "r--", linewidth=1.5, label="Perfect Prediction")
# plt.xlabel("Actual Charges (USD)")
# plt.ylabel("Predicted Charges (USD)")
# plt.title("Actual vs Predicted – Random Forest")
# plt.legend()
# plt.tight_layout()
# plt.show()

### 4.14 — Cross-Validation

In [ ]:
# TODO: 5-fold CV on the full dataset for your best model.

# cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring="r2")
# print(f"Random Forest – 5-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
✅ **Blueprint complete!** Fill in all `TODO` blocks above, run each cell, and interpret your results.
